# Hey-Function-Burmese (FunctionGemma-270M-Alpha)
## End-to-End Fine-Tuning & Multi-Tool Benchmark v2 Pipeline

This notebook contains the complete, reproducible training and evaluation pipeline for **Hey-270M-Alpha**, a lightweight on-device function-calling agent fine-tuned on Burmese, English, and code-switching natural language instructions.

- **Base Model:** [`google/functiongemma-270m-it`](https://huggingface.co/google/functiongemma-270m-it)
- **Fine-Tuned Checkpoint:** [`thantzinphyo/Hey-270M-Alpha`](https://huggingface.co/thantzinphyo/Hey-270M-Alpha)
- **Dataset:** [`thantzinphyo/Hey-Function-Burmese`](https://huggingface.co/datasets/thantzinphyo/Hey-Function-Burmese)
- **Architecture:** Full-parameter fine-tuning (All 270M weights unfrozen) with response-only loss masking and chunked cross-entropy.


In [ ]:
# [1/5] Environment Setup & Dependencies
!pip install -q -U "transformers>=4.44" "accelerate>=0.33" pyarrow huggingface_hub


In [ ]:
# [2/5] Imports, Environment Configuration & Hyperparameters
import os, gc, re, math, json, time, random
from functools import partial
from collections import defaultdict, Counter
from pathlib import Path

import torch
import torch.nn.functional as F
import pyarrow.parquet as pq
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from huggingface_hub import hf_hub_download

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Set Hugging Face Access Token (if accessing gated checkpoints or private repos)
HF_TOKEN = os.getenv("HF_TOKEN", None)

# Hyperparameters
CFG = {
    "model_id": "google/functiongemma-270m-it",
    "data_repo": "thantzinphyo/Hey-Function-Burmese",
    "output_dir": "./functiongemma-270m-alpha",
    "max_seq_len": 2800,       # 2800 tokens ensures 0.0% sequence truncation
    "epochs": 1,
    "per_dev_bs": 1,
    "grad_accum": 32,          # Effective batch size = 32
    "learning_rate": 2e-5,
    "warmup_steps": 36,
    "seed": 42,
}

torch.manual_seed(CFG["seed"])
random.seed(CFG["seed"])
print(f"CUDA available: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


In [ ]:
# [3/5] Dataset Processing with Candidate Tools Selection & Causal Collation
class FunctionGemmaDataset(torch.utils.data.Dataset):
    """
    Memory-efficient dataset that dynamically includes only the ground-truth target tools
    plus 2 negative distractor tools per sample. This prevents 61-tool schema bloat from
    exceeding VRAM and sequence boundaries.
    """
    def __init__(self, messages_list, global_tools, tokenizer, max_seq_length=2800):
        self.messages_list = messages_list
        self.tools_by_name = {t["function"]["name"]: t for t in global_tools}
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length
        self.resp_tokens = tokenizer.encode("<start_of_turn>model\n", add_special_tokens=False)

    def __len__(self):
        return len(self.messages_list)

    def __getitem__(self, idx):
        msgs = self.messages_list[idx]
        if isinstance(msgs, str):
            msgs = json.loads(msgs)
        elif hasattr(msgs, "tolist"):
            msgs = msgs.tolist()

        called = []
        for m in msgs:
            if m.get("tool_calls"):
                for tc in m["tool_calls"]:
                    n = tc.get("function", {}).get("name")
                    if n:
                        called.append(n)

        seen = set()
        sample_tools = []
        for n in called:
            if n in self.tools_by_name and n not in seen:
                sample_tools.append(self.tools_by_name[n])
                seen.add(n)
        distractors = [t for n, t in self.tools_by_name.items() if n not in seen]
        sample_tools.extend(distractors[:2])

        text = self.tokenizer.apply_chat_template(msgs, tools=sample_tools, tokenize=False)
        input_ids = self.tokenizer(text, truncation=True, max_length=self.max_seq_length, add_special_tokens=False)["input_ids"]

        # Response-only loss masking: mask user prompt and tools with -100
        labels = list(input_ids)
        r_len = len(self.resp_tokens)
        match_idx = -1
        for i in range(len(input_ids) - r_len + 1):
            if input_ids[i:i + r_len] == self.resp_tokens:
                match_idx = i + r_len
                break
        if match_idx != -1:
            labels[:match_idx] = [-100] * match_idx
        else:
            labels = [-100] * len(labels)

        return {"input_ids": input_ids, "labels": labels}

def causal_collate_fn(batch, pad_id=0):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, labels, attention_mask = [], [], []
    for x in batch:
        inp, lbl = x["input_ids"], x["labels"]
        pad_len = max_len - len(inp)
        input_ids.append(inp + [pad_id] * pad_len)
        labels.append(lbl + [-100] * pad_len)
        attention_mask.append([1] * len(inp) + [0] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
    }


In [ ]:
# [4/5] Chunked Cross-Entropy Loss & Trainer Architecture
def chunked_cross_entropy(logits, labels, chunk_size=256, ignore_index=-100, reduction="mean"):
    """
    Zero-copy chunked cross-entropy. FunctionGemma uses a 256,000 token vocabulary;
    standard F.cross_entropy on full logits would allocate >8 GB of VRAM at once.
    Chunking across sequence positions keeps peak memory under 5 GB.
    """
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    flat_logits = shift_logits.reshape(-1, shift_logits.size(-1))
    flat_labels = shift_labels.reshape(-1)
    total = (flat_labels != ignore_index).sum()
    if total == 0:
        return flat_logits.sum() * 0.0
    accum = 0.0
    for i in range(0, flat_labels.size(0), chunk_size):
        c_lbl = flat_labels[i:i + chunk_size]
        if (c_lbl != ignore_index).any():
            accum = accum + F.cross_entropy(
                flat_logits[i:i + chunk_size].float(),
                c_lbl,
                ignore_index=ignore_index,
                reduction="sum",
            )
    return accum if reduction == "sum" else accum / total

class CleanCausalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = chunked_cross_entropy(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

class LiveProgressCallback(TrainerCallback):
    def __init__(self, total_steps):
        self.total_steps = total_steps
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print("=" * 75)
        print(f"TRAINING START | Total Steps: {self.total_steps:,}")
        print("=" * 75, flush=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or "loss" not in logs: return
        step = state.global_step
        loss = logs["loss"]; lr = logs.get("learning_rate", 0.0)
        elapsed = time.time() - (self.start_time or time.time())
        sps = elapsed / max(step, 1)
        eta = (self.total_steps - step) * sps
        vram = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0
        print(f"[Step {step:4d}/{self.total_steps}] Loss: {loss:.4f} | LR: {lr:.2e} | {sps:.1f}s/step (ETA {int(eta//60)}m) | VRAM: {vram:.2f} GB", flush=True)


In [ ]:
# [5/5] Full Training Execution & Model Export
def train_model():
    print("[1/3] Downloading Hey-Function-Burmese dataset...")
    data_path = hf_hub_download(CFG["data_repo"], "train.parquet", repo_type="dataset", token=HF_TOKEN)
    tbl = pq.read_table(data_path, columns=["messages", "tools", "metadata"])
    global_tools = tbl.column("tools")[0].as_py()
    msgs_col = tbl.column("messages").to_pylist()
    meta_col = tbl.column("metadata").to_pylist()
    train_msgs = [m for m, meta in zip(msgs_col, meta_col) if meta != "eval"]
    eval_msgs = [m for m, meta in zip(msgs_col, meta_col) if meta == "eval"]
    print(f"  → Loaded: {len(train_msgs):,} train samples, {len(eval_msgs):,} eval samples, {len(global_tools)} tools")

    print(f"[2/3] Loading tokenizer and model: {CFG['model_id']}...")
    tokenizer = AutoTokenizer.from_pretrained(CFG["model_id"], token=HF_TOKEN)
    if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token

    train_ds = FunctionGemmaDataset(train_msgs, global_tools, tokenizer, CFG["max_seq_len"])
    eval_subset = random.Random(CFG["seed"]).sample(eval_msgs, min(500, len(eval_msgs)))
    eval_ds = FunctionGemmaDataset(eval_subset, global_tools, tokenizer, CFG["max_seq_len"])

    model = AutoModelForCausalLM.from_pretrained(
        CFG["model_id"], token=HF_TOKEN, torch_dtype=torch.float32, attn_implementation="eager",
    )
    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    total_steps = math.ceil(len(train_ds) / (CFG["per_dev_bs"] * CFG["grad_accum"])) * CFG["epochs"]
    training_args = TrainingArguments(
        output_dir=CFG["output_dir"],
        per_device_train_batch_size=CFG["per_dev_bs"],
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=CFG["grad_accum"],
        learning_rate=CFG["lr"],
        num_train_epochs=CFG["epochs"],
        warmup_steps=CFG["warmup_steps"],
        logging_steps=5,
        save_strategy="epoch",
        save_total_limit=1,
        fp16=True,
        report_to="none",
    )

    trainer = CleanCausalTrainer(
        model=model, args=training_args, train_dataset=train_ds, eval_dataset=eval_ds,
        data_collator=partial(causal_collate_fn, pad_id=tokenizer.pad_token_id),
        callbacks=[LiveProgressCallback(total_steps)],
    )

    print("[3/3] Training...")
    trainer.train()
    trainer.save_model(CFG["output_dir"])
    tokenizer.save_pretrained(CFG["output_dir"])
    print(f"SUCCESS: Fine-tuned model and tokenizer saved cleanly to '{CFG['output_dir']}'!")

# To execute full training, uncomment the following line:
# train_model()


## Benchmark v2: Multi-Tool Aware Evaluation

The following evaluation cell performs multi-tool aware benchmark testing on held-out evaluation samples across:
1. **English Queries**
2. **Burmese Queries**
3. **Code-Switching Queries**

Metrics reported:
- **Fn Acc (Function Accuracy):** All predicted function names match ground truth.
- **Arg Acc (Argument Accuracy):** Predicted function names and all extracted arguments match ground truth.
- **Exact Match (EM):** Strict end-to-end match.
- **Per-Tool Accuracy Table & Confusion Matrix**


In [ ]:
# ============================================================
# FunctionGemma Burmese & Code-Switch Benchmark v2
# ============================================================
BURMESE_RANGES = [(0x1000, 0x109F), (0xA9E0, 0xA9FF), (0xAA60, 0xAA7F)]
def _has_burmese(s): return any(any(lo <= ord(c) <= hi for lo, hi in BURMESE_RANGES) for c in s)
def _has_latin(s): return any(65 <= ord(c) <= 90 or 97 <= ord(c) <= 122 for c in s)
def classify_register(text):
    b, e = _has_burmese(text), _has_latin(text)
    if b and e: return "Code-Switch"
    if b: return "Burmese"
    if e: return "English"
    return "Other"
def last_user_text(msgs):
    for m in reversed(msgs):
        if m.get("role") == "user":
            c = m.get("content", ""); return c if isinstance(c, str) else str(c)
    return ""

def _parse_fm_args(s):
    args = {}
    i, n = 0, len(s)
    while i < n:
        while i < n and s[i] in ', \n\t': i += 1
        if i >= n: break
        c = s.find(':', i)
        if c < 0: break
        key = s[i:c].strip(); i = c + 1
        if s.startswith('<escape>', i):
            e = s.find('<escape>', i + 8)
            if e < 0: args[key] = s[i + 8:]; i = n
            else: args[key] = s[i + 8:e]; i = e + 8
        else:
            e = s.find(',', i)
            if e < 0: e = n
            raw = s[i:e].strip(); i = e
            if raw in ('None', ''): args[key] = None
            elif raw in ('True', 'true'): args[key] = True
            elif raw in ('False', 'false'): args[key] = False
            else:
                try: args[key] = float(raw) if '.' in raw else int(raw)
                except: args[key] = raw
    return args

def extract_all_tool_calls(text):
    if not text or not text.strip(): return []
    return [{"name": m.group(1), "arguments": _parse_fm_args(m.group(2))}
            for m in re.finditer(r'<start_function_call>call:(\w+)\{(.*?)\}<end_function_call>', text, re.DOTALL)]

def _clean_args(d):
    return {k: v for k, v in d.items() if v is not None} if isinstance(d, dict) else {}

def _val_eq(v1, v2):
    s1, s2 = str(v1).strip().lower(), str(v2).strip().lower()
    if s1 == s2: return True
    try: return float(s1) == float(s2)
    except: return False

def compare_args(gt, pred):
    g = _clean_args(gt); p = _clean_args(pred)
    if set(g.keys()) != set(p.keys()): return False
    return all(_val_eq(p[k], g[k]) for k in g)

def run_benchmark_v2(model_checkpoint="thantzinphyo/Hey-270M-Alpha", n_bench=200):
    print(f"[load] Loading evaluation checkpoint: {model_checkpoint}...")
    tok = AutoTokenizer.from_pretrained(model_checkpoint, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        model_checkpoint, token=HF_TOKEN, torch_dtype=torch.float32, attn_implementation="eager",
    ).cuda().eval()

    data_path = hf_hub_download(CFG["data_repo"], "train.parquet", repo_type="dataset", token=HF_TOKEN)
    t = pq.read_table(data_path, columns=["messages", "tools", "metadata"])
    global_tools = t.column("tools")[0].as_py()
    tools_by_name = {x["function"]["name"]: x for x in global_tools}
    msgs_col = t.column("messages").to_pylist()
    meta_col = t.column("metadata").to_pylist()
    eval_msgs = [m for m, x in zip(msgs_col, meta_col) if x == "eval"]

    parsed = [json.loads(m) if isinstance(m, str) else (m.tolist() if hasattr(m, 'tolist') else m) for m in eval_msgs]
    registers = [classify_register(last_user_text(m)) for m in parsed]
    by_reg = defaultdict(list)
    for i, r in enumerate(registers): by_reg[r].append(i)
    sample_indices = []
    for r, idxs in by_reg.items():
        n_take = max(1, round(n_bench * len(idxs) / len(registers)))
        sample_indices.extend(random.sample(idxs, min(n_take, len(idxs))))
    sample_indices = sample_indices[:n_bench]
    random.shuffle(sample_indices)
    print(f"[bench] Running Benchmark v2 on {len(sample_indices)} held-out samples...")

    results = defaultdict(lambda: {"total": 0, "exact": 0, "wrong_args": 0, "wrong_tool": 0})
    overall = {"total": 0, "exact": 0, "wrong_args": 0, "wrong_tool": 0}
    per_tool = defaultdict(lambda: {"total": 0, "ok": 0})

    for k, idx in enumerate(sample_indices):
        msgs = parsed[idx]; reg = registers[idx]
        gt_calls = []
        for m in msgs:
            if m.get("tool_calls"):
                for tc in m["tool_calls"]: gt_calls.append(tc["function"])
        if not gt_calls: continue
        gt_names = [c["name"] for c in gt_calls]
        seen = set(); st = []
        for nm in gt_names:
            if nm in tools_by_name and nm not in seen:
                st.append(tools_by_name[nm]); seen.add(nm)
        st.extend([x for kk, x in tools_by_name.items() if kk not in seen][:2])

        prompt_msgs = [m for m in msgs if m.get("role") != "assistant"]
        prompt = tok.apply_chat_template(prompt_msgs, tools=st, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=2800).to("cuda")

        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=300, do_sample=False, pad_token_id=tok.pad_token_id)
        gen_raw = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
        pred_calls = extract_all_tool_calls(gen_raw)

        if not pred_calls:
            cat = "wrong_tool"
        else:
            pred_names = [c["name"] for c in pred_calls]
            if set(pred_names) != set(gt_names):
                cat = "wrong_tool"
            else:
                all_ok = all(
                    next((compare_args(gt["arguments"], p["arguments"]) for p in pred_calls if p["name"] == gt["name"]), False)
                    for gt in gt_calls
                )
                cat = "exact" if all_ok else "wrong_args"

        results[reg]["total"] += 1; overall["total"] += 1
        results[reg][cat] += 1; overall[cat] += 1

        for gt in gt_calls:
            nm = gt["name"]; per_tool[nm]["total"] += 1
            match = next((p for p in pred_calls if p["name"] == nm), None)
            if match and compare_args(gt["arguments"], match["arguments"]):
                per_tool[nm]["ok"] += 1

    def pct(n, d): return (n / d * 100) if d else 0.0
    print("\n" + "=" * 70)
    print(f"Evaluation Benchmark v2 (Held-out Set: {overall['total']} samples)")
    print("=" * 70)
    print(f"{'Register':<14}{'Samples':>8}   {'Fn Acc':>8}   {'Arg Acc':>9}   {'EM':>7}")
    print("-" * 70)
    for reg_name in ["English", "Burmese", "Code-Switch"]:
        r = results[reg_name]
        if r["total"] == 0: continue
        fn_ok = r["exact"] + r["wrong_args"]
        print(f"{reg_name:<14}{r['total']:>8}   {pct(fn_ok, r['total']):>7.1f}%   {pct(r['exact'], r['total']):>8.1f}%   {pct(r['exact'], r['total']):>6.1f}%")
    print("-" * 70)
    fn_all = overall["exact"] + overall["wrong_args"]
    print(f"{'Overall':<14}{overall['total']:>8}   {pct(fn_all, overall['total']):>7.1f}%   {pct(overall['exact'], overall['total']):>8.1f}%   {pct(overall['exact'], overall['total']):>6.1f}%")

# Run benchmark directly against published weights
run_benchmark_v2("thantzinphyo/Hey-270M-Alpha")
